In [1]:
import pandas as pd
from deployment_utils import emptyStateRecord, arrHourList, data_format_convertion, unixTime, TOU_15min, convertOutput, NumpyEncoder,convert_floats_to_decimals, currentTime15min, table_to_optimizedStates, stringify_keys, table_to_stateRecord
import optimizer_station_V2 as opt
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import matplotlib
import os 
import sys 

CVXPY version: 1.3.0


# Merging Station Level Optimizer with Power Dataframe

## Load in the sessions dataframe

In [2]:
sessions_df = pd.read_csv("Sessions3.csv")
sessions_df.head(3)

,dcosId,userId,vehicle_model,vehicle_maxChgRate_W,siteId,stationId,connectTime,startChargeTime,Deadline,energyReq_Wh,...,sch_centsPerOverstayHr,Duration,DurationHrs,choice,regular,scheduled,cumEnergy_Wh,peakPower_W,power,lastUpdate
0,24,605,500e,6600,23,7,2020-11-05T10:30:16,2020-11-05T10:31:09,NaN,NaN,...,200.0,0 days 03:43:57,3.73249,REGULAR,1,0,3281.0,6335,"[{'power_W': Decimal('6259'), 'timestamp': Dec...",2020-11-05T14:15:06
1,26,486,Model 3,24000,23,3,2020-11-11T07:39:55,2020-11-11T07:39:59,NaN,NaN,...,200.0,0 days 06:50:07,6.83527,REGULAR,1,0,33458.0,7005,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2020-11-11T14:30:06
2,30,620,Volt,3600,25,12,2020-11-13T16:19:55,2020-11-13T16:20:06,2020-11-14T04:15:00,18400.0,...,300.0,0 days 20:40:02,20.66722,SCHEDULED,0,1,15216.0,3450,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2020-11-14T13:00:08


## Load in the power dataframe

In [7]:
power_df = pd.read_csv("power_df_2008-2406.csv")
power_df = power_df.iloc[::3] # Change granularity from 5 minutes to 15 minutes
power_df['recordTimestamp'] = pd.to_datetime(power_df['recordTimestamp']) # Convert to timezone-naive datetime objects
power_df.head(3)

,Unnamed: 0,recordTimestamp,siteId,activeStationIds,totalPower,numberOfActiveSessions,fractionOfRegularSessions,averageRegularPricePerHour,averageScheduledPricePerHour,power1,...,dcosId4,dcosId5,dcosId6,dcosId7,dcosId8,isArtificialData,dayNumber,isWeekend,weekNumber,isHoliday
0,0,2020-08-18 18:30:00-07:00,25.0,[],0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,1,False,34,False
3,3,2020-08-18 18:45:00-07:00,25.0,[],0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,1,False,34,False
6,6,2020-08-18 19:00:00-07:00,25.0,[],0.0,0.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,False,1,False,34,False


## Running a simulation

Suppose a car arrives at 2021-03-25 at 11am. Let's look at the power dataframe at this time:

In [8]:
row = power_df.loc[62982]
row

Unnamed: 0                                          62982
recordTimestamp                 2021-03-25 11:00:00-07:00
siteId                                               25.0
activeStationIds                             [7, 3, 1, 2]
totalPower                                        15760.0
numberOfActiveSessions                                4.0
fractionOfRegularSessions                            0.25
averageRegularPricePerHour                            NaN
averageScheduledPricePerHour                          NaN
power1                                             1608.0
power2                                             6802.0
power3                                             6707.0
power4                                                NaN
power5                                                NaN
power6                                                NaN
power7                                              643.0
power8                                                NaN
dcosId1       

Now let's look up these sessions using the dcosID keys.

In [9]:
sessions_df[sessions_df["dcosId"].isin([260, 261, 262, 263])]

,dcosId,userId,vehicle_model,vehicle_maxChgRate_W,siteId,stationId,connectTime,startChargeTime,Deadline,energyReq_Wh,...,sch_centsPerOverstayHr,Duration,DurationHrs,choice,regular,scheduled,cumEnergy_Wh,peakPower_W,power,lastUpdate
155,260,673,Prius Prime,3300,25,17,2021-03-25T08:41:03,2021-03-25T08:41:15,2021-03-25T12:30:00,8800.0,...,400.0,0 days 03:33:53,3.56472,SCHEDULED,0,1,4743.0,3334,"[{'power_W': Decimal('3334'), 'timestamp': Dec...",2021-03-25T12:15:08
156,261,627,Kona,75000,25,13,2021-03-25T08:53:55,2021-03-25T08:54:02,NaN,NaN,...,500.0,0 days 04:06:07,4.10194,REGULAR,1,0,24794.0,6748,"[{'power_W': Decimal('4517'), 'timestamp': Dec...",2021-03-25T13:00:09
157,262,721,Model X,5000,25,11,2021-03-25T08:53:58,2021-03-25T08:55:16,2021-03-25T11:00:00,10667.0,...,300.0,0 days 02:04:53,2.08138,SCHEDULED,0,1,8659.0,6552,"[{'power_W': Decimal('6504'), 'timestamp': Dec...",2021-03-25T11:00:09
158,263,746,Model 3,11500,25,12,2021-03-25T09:54:22,2021-03-25T09:54:59,2021-03-25T18:00:00,36935.0,...,500.0,0 days 03:05:10,3.08611,SCHEDULED,0,1,19903.0,6829,"[{'power_W': Decimal('0'), 'timestamp': Decima...",2021-03-25T13:00:09


## Script 1: Convert Active Sessions Directly into StateRecords and use data_format_convertion

In [10]:
from deployment_utils import emptyStateRecord,currentTime15min,TOU_15min,data_format_convertion, unixTime

hour = 11
delta_t = 0.25

# Making up some parameters for our new arrival
e_need = 10 # in units of kWh
duration_hour = 6.25

event = {
    "time": int(hour / delta_t), 
    "e_need": e_need,
    "duration": duration_hour,
    "station_pow_max": 6.6,
    "user_power_rate": 6.6,
    "limit_reg_with_sch": False,
    "limit_sch_with_constant": False,
    "sch_limit": 0,
    "historical_peak": 35, # made up this number for purposes of simulation 
}

activeStations = row['activeStationIds']
print(activeStations )

# The below script uses the power dataframe to lookup needed information about active sessions
sessions = []
for activeStation in eval(activeStations):
    cosId = int(row['dcosId' + str(activeStation)])
    session = sessions_df[sessions_df['dcosId'] == cosId]

    choice = session['choice'].iloc[0][:3]

    start = pd.to_datetime(session['startChargeTime']).iloc[0]
    end = pd.to_datetime(session['Deadline']).iloc[0] if choice == "SCHEDULED" else pd.to_datetime(session['lastUpdate']).iloc[0] # For regular sessions, deadline is NaN so subsitute with lastUpdate

    power_profile = power_df[(power_df['recordTimestamp'] >= start) & (power_df['recordTimestamp'] <= end)]
    power_profile = power_profile[power_profile['recordTimestamp'].dt.minute % 15 == 0]['power' + str(activeStation)].to_numpy().reshape(-1, 1)

    sessions.append(
        {
            "dcosId" : cosId,
            "choice": choice,
            "powerRate": "HIGH",
            "energyNeeded" : session['energyReq_Wh'].iloc[0],
            "deadline" : unixTime(end) if choice == "SCHEDULED" else 0, # deadlone only applies for scheduled users
            "optPower" : power_profile 
        } 
    )

stateRecord = [
    {"monthlyPeak" : 35, 
     "timestamp" : unixTime(pd.Timestamp(year=2021, month=3, day=21, hour=11)), ## Last record TS(decision of the last vehicle)
     "sessions": sessions
    }
]

station_info = data_format_convertion(stateRecord, opt_hour=11)

par = opt.Parameters(z0 = np.array([20, 30, 1, 1]).reshape(4, 1),
         Ts = delta_t,
         eff = 1.0,
         soft_v_eta = 1e-4,
         opt_eps = 0.0001,
         TOU = TOU_15min(),
         demand_charge_cost=1800/30) ## cents/ hr 

prb = opt.Problem(par = par, event = event, station_info = station_info, k=event["time"]) # k corresponds to hour of the day


obj = opt.Optimization_station(par, prb, hour)
station_info, res = obj.run_opt("BCD") 

[7, 3, 1, 2]


TypeError: can't compare offset-naive and offset-aware datetimes

## Script 2: Convert Active Sessions Directly into session_info

In [7]:
from deployment_utils import emptyStateRecord,currentTime15min,TOU_15min,data_format_convertion, unixTime

hour = 11
delta_t = 0.25

# Making up some parameters for our new arrival
e_need = 10 # in units of kWh
duration_hour = 6.25

event = {
    "time": int(hour / delta_t), 
    "e_need": e_need,
    "duration": duration_hour,
    "station_pow_max": 6.6,
    "user_power_rate": 6.6,
    "limit_reg_with_sch": False,
    "limit_sch_with_constant": False,
    "sch_limit": 0,
    "historical_peak": 35, # made up this number for purposes of simulation 
}

activeStations = row['activeStationIds']

# The below script uses the power dataframe to lookup needed information about active sessions
sessions = []
for activeStation in eval(activeStations):
    cosId = int(row['dcosId' + str(activeStation)])
    session = sessions_df[sessions_df['dcosId'] == cosId]

    choice = session['choice'].iloc[0][:3]

    start = pd.to_datetime(session['startChargeTime']).iloc[0]
    end = pd.to_datetime(session['Deadline']).iloc[0] if choice == "SCHEDULED" else pd.to_datetime(session['lastUpdate']).iloc[0] # For regular sessions, deadline is NaN so subsitute with lastUpdate

    power_profile = power_df[(power_df['recordTimestamp'] >= start) & (power_df['recordTimestamp'] <= end)]
    power_profile = power_profile[power_profile['recordTimestamp'].dt.minute % 15 == 0]['power' + str(activeStation)].to_numpy().reshape(-1, 1)

    station_info.append(
        {'choice' : choice[:3], 
         'dcosId' : session['dcosId'].iloc[0], 
         "start_time" : pd.to_datetime(session['startChargeTime']).iloc[0].hour, 
         "end_time" :  pd.to_datetime(session['startChargeTime']).iloc[0].hour + session['DurationHrs'], # 
         "energyNeeded" : session['energyReq_Wh'].iloc[0] if choice== "SCHEDULED" else 6.6 * session['DurationHrs'].iloc[0], 
         "optPower" : power_profile, 
         "power_rate" : 6.6, 
         "price" : session['reg_centsPerHr'] if choice== "REGULAR" else session['sch_centsPerHr'], 
         "TOU_idx" : 0
        }
    )


par = opt.Parameters(z0 = np.array([20, 30, 1, 1]).reshape(4, 1),
         Ts = delta_t,
         eff = 1.0,
         soft_v_eta = 1e-4,
         opt_eps = 0.0001,
         TOU = TOU_15min(),
         demand_charge_cost=1800/30) ## cents/ hr 

prb = opt.Problem(par = par, event = event, station_info = station_info, k=event["time"]) # k corresponds to hour of the day


obj = opt.Optimization_station(par, prb, hour)
station_info, res = obj.run_opt("BCD") 

ValueError: Length of values (0) does not match length of index (1)